In [ ]:
# Pressure differences

import logging
import sys
from pathlib import Path

import pandas as pd
from tqdm import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", stream=sys.stdout)
logger = logging.getLogger()

output_base_dir = Path("output")
pressure_diff_dir = output_base_dir / "Pressure_Differences"
pressure_diff_dir.mkdir(parents=True, exist_ok=True)

leak_pressure_dir = output_base_dir / "leak" / "Leak_Pressure"
baseline_pressure_dir = output_base_dir / "baseline" / "Baseline_Pressure"

batch_pairs = []
for leak_file in sorted(leak_pressure_dir.glob("Leak_Pressure_batch*.parquet")):
    batch_id = leak_file.stem.split("batch")[1]
    baseline_file = baseline_pressure_dir / f"Baseline_Pressure_batch{batch_id}.parquet"
    if baseline_file.exists():
        batch_pairs.append((leak_file, baseline_file, batch_id))

if not batch_pairs:
    raise FileNotFoundError(
        f"No matching leak/baseline batch pairs found under {leak_pressure_dir} and {baseline_pressure_dir}"
    )

for leak_file, baseline_file, batch_id in tqdm(batch_pairs, desc="Processing batches"):
    output_file = pressure_diff_dir / f"Pressure_Diffs_batch{batch_id}.parquet"
    if output_file.exists():
        continue

    try:
        sample = pd.read_parquet(baseline_file, engine="pyarrow").head(1)
        node_columns = [col for col in sample.columns if col != "Iteration"]

        leak_df_iter = pd.read_parquet(leak_file, columns=["Iteration"], engine="pyarrow")
        baseline_df_iter = pd.read_parquet(
            baseline_file, columns=["Iteration"], engine="pyarrow"
        )

        common_iters = sorted(
            set(leak_df_iter["Iteration"].unique()).intersection(
                set(baseline_df_iter["Iteration"].unique())
            )
        )

        chunk_size = 500
        all_pressure_diffs = []

        for i in range(0, len(common_iters), chunk_size):
            chunk_iters = common_iters[i : i + chunk_size]
            if not chunk_iters:
                continue

            leak_data = pd.read_parquet(
                leak_file,
                engine="pyarrow",
                columns=["Iteration"] + node_columns,
                filters=[("Iteration", "in", chunk_iters)],
            )
            baseline_data = pd.read_parquet(
                baseline_file,
                engine="pyarrow",
                filters=[("Iteration", "in", chunk_iters)],
            )

            leak_data.sort_values("Iteration", inplace=True)
            baseline_data.sort_values("Iteration", inplace=True)

            pressure_diffs = (leak_data[node_columns] - baseline_data[node_columns]).abs()
            pressure_diffs.insert(0, "Iteration", leak_data["Iteration"].astype("int64"))

            all_pressure_diffs.append(pressure_diffs)
            del leak_data, baseline_data, pressure_diffs

        if all_pressure_diffs:
            combined_diffs = pd.concat(all_pressure_diffs, ignore_index=True)
            combined_diffs["Iteration"] = combined_diffs["Iteration"].astype("int64")
            combined_diffs.sort_values("Iteration", inplace=True)

            combined_diffs.to_parquet(
                output_file,
                engine="pyarrow",
                compression="snappy",
                index=False,
            )

            del combined_diffs, all_pressure_diffs

    except Exception as e:
        logger.error(f"Error processing batch {batch_id}: {e}")
        import traceback

        logger.error(traceback.format_exc())

logger.info("Merging pressure difference batches...")
merged_file = pressure_diff_dir / "Pressure_Differences_Matrix.parquet"
diff_batch_files = sorted(pressure_diff_dir.glob("Pressure_Diffs_batch*.parquet"))

if not diff_batch_files:
    raise FileNotFoundError(f"No pressure difference batch files found in {pressure_diff_dir}")

try:
    if len(diff_batch_files) <= 10:
        all_dfs = []
        for batch_file in tqdm(diff_batch_files, desc="Reading batch files"):
            df = pd.read_parquet(batch_file, engine="pyarrow")
            df["Iteration"] = df["Iteration"].astype("int64")
            all_dfs.append(df)

        if not all_dfs:
            raise RuntimeError("No dataframes to merge")

        combined = pd.concat(all_dfs, ignore_index=True)
        combined.to_parquet(
            merged_file,
            engine="pyarrow",
            compression="snappy",
            index=False,
        )
        del combined, all_dfs

    else:
        import shutil

        first_file = diff_batch_files[0]
        shutil.copy(str(first_file), str(merged_file))

        chunk_size = 5
        remaining_files = diff_batch_files[1:]

        with tqdm(total=len(remaining_files), desc="Merging batch files") as pbar:
            for i in range(0, len(remaining_files), chunk_size):
                chunk_files = remaining_files[i : i + chunk_size]

                merged_df = pd.read_parquet(merged_file, engine="pyarrow")
                merged_df["Iteration"] = merged_df["Iteration"].astype("int64")

                for batch_file in chunk_files:
                    batch_df = pd.read_parquet(batch_file, engine="pyarrow")
                    batch_df["Iteration"] = batch_df["Iteration"].astype("int64")
                    merged_df = pd.concat([merged_df, batch_df], ignore_index=True)
                    del batch_df
                    pbar.update(1)

                merged_df.to_parquet(
                    merged_file,
                    engine="pyarrow",
                    compression="snappy",
                    index=False,
                )
                del merged_df

except Exception as e:
    logger.error(f"Error during merge process: {e}")
    import traceback

    logger.error(traceback.format_exc())

In [ ]:
# Sigmoid transform

import logging
import sys
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from tqdm import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", stream=sys.stdout)
logger = logging.getLogger()


def apply_sigmoid_transformation(df, threshold_center=1.05, steepness=4.88):
    data_cols = [c for c in df.columns if c != "Iteration"]
    result = df.copy()
    result[data_cols] = 1 / (1 + np.exp(-steepness * (df[data_cols] - threshold_center)))
    return result


def process_chunk(input_file, iterations_in_chunk, output_file, threshold_center, steepness):
    chunk = pd.read_parquet(input_file, filters=[("Iteration", "in", iterations_in_chunk)])
    if len(chunk) == 0:
        return 0

    transformed = apply_sigmoid_transformation(
        chunk, threshold_center=threshold_center, steepness=steepness
    )
    transformed.to_parquet(
        output_file,
        engine="pyarrow",
        compression="snappy",
        index=False,
    )
    return len(chunk)


def transform_pressure_differences(
    input_file: Path,
    output_file: Path,
    threshold_center=1.05,
    steepness=4.88,
    chunk_size=5000,
    max_workers=1,
):
    output_file.parent.mkdir(parents=True, exist_ok=True)
    
    temp_dir = output_file.parent / "temp"
    temp_dir.mkdir(parents=True, exist_ok=True)

    if not input_file.exists():
        logger.error(f"Input file not found: {input_file}")
        return False

    try:
        df_metadata = pd.read_parquet(input_file, columns=["Iteration"])
        all_iterations = sorted(df_metadata["Iteration"].unique())

        chunk_ranges = []
        for i in range(0, len(all_iterations), chunk_size):
            iterations_in_chunk = all_iterations[i : i + chunk_size]
            if iterations_in_chunk:
                chunk_ranges.append(iterations_in_chunk)

        chunk_files = [
            temp_dir / f"sigmoid_chunk_{i}.parquet"
            for i in range(len(chunk_ranges))
        ]

        with tqdm(total=len(df_metadata), desc="Applying sigmoid transformation") as pbar:
            if max_workers > 1:
                processed_rows_lock = threading.Lock()

                def process_and_update(iterations_in_chunk, chunk_file):
                    rows = process_chunk(
                        input_file,
                        iterations_in_chunk,
                        chunk_file,
                        threshold_center,
                        steepness,
                    )
                    with processed_rows_lock:
                        pbar.update(rows)
                    return rows

                with ThreadPoolExecutor(max_workers=max_workers) as executor:
                    futures = [
                        executor.submit(process_and_update, iters, f)
                        for iters, f in zip(chunk_ranges, chunk_files)
                    ]
                    for future in as_completed(futures):
                        future.result()
            else:
                for iterations_in_chunk, temp_file in zip(chunk_ranges, chunk_files):
                    processed_rows = process_chunk(
                        input_file,
                        iterations_in_chunk,
                        temp_file,
                        threshold_center,
                        steepness,
                    )
                    pbar.update(processed_rows)

        valid_chunk_files = []
        for chunk_file in chunk_files:
            if chunk_file.exists() and chunk_file.stat().st_size > 0:
                try:
                    pq.read_metadata(chunk_file)
                    valid_chunk_files.append(chunk_file)
                except Exception:
                    continue

        all_dfs = []
        for chunk_file in tqdm(valid_chunk_files, desc="Reading chunks"):
            all_dfs.append(pd.read_parquet(chunk_file))

        if all_dfs:
            final_df = pd.concat(all_dfs, ignore_index=True)
            final_df = final_df.drop_duplicates(subset=["Iteration"]).sort_values("Iteration")
            final_df.to_parquet(
                output_file,
                engine="pyarrow",
                compression="snappy",
                index=False,
            )

        for chunk_file in chunk_files:
            if chunk_file.exists():
                try:
                    chunk_file.unlink()
                except Exception:
                    pass

        try:
            if temp_dir.exists() and not any(temp_dir.iterdir()):
                temp_dir.rmdir()
        except Exception:
            pass

        logger.info(f"Wrote: {output_file}")
        return True

    except Exception as e:
        logger.error(f"Error during sigmoid transformation: {e}")
        import traceback

        logger.error(traceback.format_exc())
        return False


if __name__ == "__main__":
    output_base_dir = Path("output")
    transform_pressure_differences(
        input_file=output_base_dir / "Pressure_Differences" / "Pressure_Differences_Matrix.parquet",
        output_file=output_base_dir / "Sigmoid_Matrix" / "Sigmoid_Detection_Matrix.parquet",
        threshold_center=1.05,
        steepness=4.88,
        chunk_size=10000,
        max_workers=1,
    )

In [ ]:
# Sigmoid matrix post-process (drop columns to remove reservoirs and tanks)

import logging
import sys
from pathlib import Path

import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", stream=sys.stdout)
logger = logging.getLogger()

output_base_dir = Path("output")
input_parquet_file = output_base_dir / "Sigmoid_Matrix" / "Sigmoid_Detection_Matrix.parquet"
output_parquet_file = output_base_dir / "Sigmoid_Matrix" / "Sigmoid_Detection_Matrix_Processed.parquet"

columns_to_remove = ['1']  # Hanoi
# columns_to_remove = ['R1', 'T1', 'T2', 'T3', 'T4', 'T5', 'T6', 'T7']  # C-Town

try:
    if not input_parquet_file.exists():
        raise FileNotFoundError(f"File not found: {input_parquet_file}")

    df = pd.read_parquet(input_parquet_file, engine="pyarrow")

    for col in columns_to_remove:
        if col in df.columns:
            df.drop(columns=col, inplace=True)

    df["Iteration"] = df["Iteration"].astype("int64")

    df.to_parquet(
        output_parquet_file,
        engine="pyarrow",
        compression="snappy",
        index=False,
    )

    logger.info(f"Wrote: {output_parquet_file}")

except Exception as e:
    logger.error(f"An error occurred: {e}")
    import traceback

    logger.error(traceback.format_exc())
    raise

In [ ]:
# Distance matrix

import logging
import sys
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
import wntr

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", stream=sys.stdout)
logger = logging.getLogger()

output_base_dir = Path("output")
inp_file = Path("networks/Hanoi.inp") # Hanoi.inp, C-Town.inp
processed_sigmoid_file = output_base_dir / "Sigmoid_Matrix" / "Sigmoid_Detection_Matrix_Processed.parquet"

output_parquet_dir = output_base_dir / "Distance_Matrix"
output_parquet_dir.mkdir(parents=True, exist_ok=True)
output_parquet_file = output_parquet_dir / "Distance_Matrix_Processed.parquet"

wn = wntr.network.WaterNetworkModel(str(inp_file))

G = wn.to_graph()
uG = G.to_undirected()

distances = dict(nx.all_pairs_dijkstra_path_length(uG))
distance_df = pd.DataFrame(
    distances, index=wn.node_name_list, columns=wn.node_name_list
).fillna(float("inf"))

if not processed_sigmoid_file.exists():
    raise FileNotFoundError(f"Processed sigmoid file not found at: {processed_sigmoid_file}")

sigmoid_df = pd.read_parquet(processed_sigmoid_file, engine="pyarrow").head(1)
node_names = [col for col in sigmoid_df.columns if col != "Iteration"]
if not node_names:
    raise RuntimeError(f"No node columns found in {processed_sigmoid_file}")

distance_df = distance_df.reindex(index=node_names, columns=node_names, fill_value=np.inf)

distance_df.to_parquet(
    output_parquet_file,
    engine="pyarrow",
    compression="snappy",
    index=True,
)

logger.info(f"Wrote: {output_parquet_file}")